# Per-store FP-Growth experiment

Validate the per-store basket-building and frequent-itemset flow against the read-only SQL Server warehouse.


## Imports

Load only the packages used by this experiment.


In [ ]:
import os
import pyodbc
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from fpgrowth_py import fpgrowth
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


## Configure the read-only SQL Server source

Load credentials from the project environment and verify the ODBC connection.


In [ ]:
# Load environment variables from the project root when available.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

datasets_dir = project_root / "datasets"
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()

dw_user = (os.getenv("DATAWAREHOUSE_USER") or "").strip()
dw_password = (os.getenv("DATAWAREHOUSE_PASSWORD") or "").strip()
dw_host = (os.getenv("DATAWAREHOUSE_HOST") or "").strip()
dw_database = (os.getenv("DATAWAREHOUSE_DATABASE") or "").strip()

if not all([dw_user, dw_password, dw_host, dw_database]):
    raise ValueError("Missing one or more DATAWAREHOUSE_* environment variables.")

drivers = pyodbc.drivers()
print("Available ODBC drivers:", drivers)

preferred_driver_names = [
    (os.getenv("SQLSERVER_DRIVER") or "").strip(),
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server",
]

driver = next(
    (
        candidate
        for candidate in preferred_driver_names
        if candidate and any(candidate.lower() in d.lower() for d in drivers)
    ),
    None,
)

if not driver:
    raise RuntimeError(
        f"No supported SQL Server ODBC driver found. Available drivers: {drivers}. "
        "Install Microsoft ODBC Driver 18 for SQL Server and verify the 64-bit ODBC Administrator."
    )

print(f"Using SQL Server driver: {driver}")

connection_string = URL.create(
    drivername="mssql+pyodbc",
    username=dw_user,
    password=dw_password,
    host=dw_host,
    database=dw_database,
    query={
        "driver": driver,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
        "LoginTimeout": "30",
    },
)

engine = create_engine(connection_string, pool_pre_ping=True, future=True)

with engine.connect() as conn:
    print("Database connection OK:", conn.execute(text("SELECT 1")).scalar())

query = text(
    """
    SELECT 
        dd.[date] AS TRX_date,
        [StoreCode],
        [BillNo],
        [ItemCode],
        ITEMLONGNAME,
        [Quantity],
        DEPARTMENT,
        CLASS,
        SUBCLASS,
        [UOM_CD],
        [TotalAmt],
        [NetValue],
        [WACValue],
        [CSM_QTY],
        [CONSIGN_FINAL_QTY],
        [POS_FINAL_QTY]
    FROM [DBWH_8555].[dbo].[FactSalesTrxNew] fstn
    INNER JOIN dimdate dd ON dd.datekey = fstn.DateKey
    INNER JOIN DimItem di ON fstn.ItemCode = di.ITMCD
    WHERE dd.[date] BETWEEN :start_date AND :end_date AND StoreCode = '083'
    """)


## Load and basketize transactions

Extract one store-period and convert each bill into a unique item basket.


In [ ]:
with engine.connect() as conn:
    start_date = '2025-11-01'
    end_date = '2025-11-30'

    store_df = pd.read_sql(query, conn,  params={'start_date': start_date, 'end_date': end_date})

transactions = (
    store_df.groupby("BillNo")["ItemCode"]
    .apply(lambda x: list(set(x)))  # remove duplicates
    .apply(list).tolist()
)
n_transactions = len(transactions)

print(f"Number of transactions: {n_transactions}")

## Validate basket quality

Review transaction and item counts before mining.


In [ ]:
print("Total transactions:", len(transactions))

unique_items = set(item for t in transactions for item in t)
print("Unique products:", len(unique_items))

max_items_per_bill = max(len(t) for t in transactions)
print("Max items per transaction:", max_items_per_bill)

avg_items = sum(len(t) for t in transactions) / len(transactions)
print("Average items per transaction:", avg_items)

In [ ]:
# transactions.to_csv(datasets_dir / "transaction_test.csv", index=False)
transactions

## Mine frequent itemsets

Run FP-Growth using explicit experimental thresholds.


In [ ]:
fp_result = fpgrowth(transactions, minSupRatio=0.1, minConf=0.5)

if fp_result is None:
    raise RuntimeError("fpgrowth returned no result. Check the transaction data or support/confidence thresholds.")

freqItemSet, rules_fpg = fp_result
transactions_sets = [set(t) for t in transactions]  # convert once, reuse


In [ ]:
freqItemSet

In [ ]:
rules_fpg

## Recalculate and normalize support

Convert the library result into the dataframe contract expected by association-rule tooling.


In [ ]:
def calc_support(itemset):
    return sum(1 for t in transactions_sets if itemset.issubset(t)) / n_transactions

frequent_itemsets = pd.DataFrame([
    {"itemsets": frozenset(itemset), "support": calc_support(itemset)}
    for itemset in freqItemSet
])

frequent_itemsets.head(3)


In [ ]:
# Ensure column dtypes
frequent_itemsets["itemsets"] = frequent_itemsets["itemsets"].apply(frozenset)
frequent_itemsets = frequent_itemsets.drop_duplicates(subset=["itemsets"]).reset_index(drop=True)
frequent_itemsets.head(3)